In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [2]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Load cleaned data
df = pd.read_csv('cleaned_tweets.csv')
df = df.dropna(subset=['clean_text'])

# Convert labels to numbers
# negative=0, neutral=1, positive=2
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['airline_sentiment'].map(label_map)

print("Labels sample:")
print(df[['airline_sentiment', 'label']].head())

Labels sample:
  airline_sentiment  label
0           neutral      1
1          positive      2
2           neutral      1
3          negative      0
4          negative      0


In [3]:
# Tokenization
MAX_WORDS = 10000  # vocabulary size
MAX_LEN = 50       # max words per tweet

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'])

# Convert words to sequences of numbers
sequences = tokenizer.texts_to_sequences(df['clean_text'])

# Pad so all tweets are same length
padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print("Vocabulary size:", len(tokenizer.word_index))
print("Padded shape:", padded.shape)
print("\nExample tweet:")
print("Text:   ", df['clean_text'][1])
print("Numbers:", padded[1])

Vocabulary size: 10153
Padded shape: (14614, 50)

Example tweet:
Text:    plus youve added commercial experience tacky
Numbers: [ 407  408  941 1059  108 4951    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]


In [4]:
X = padded
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)

X_train shape: (11691, 50)
X_test shape:  (2923, 50)
y_train shape: (11691,)
y_test shape:  (2923,)


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')  # 3 classes: negative/neutral/positive
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

C:\Users\HP\anaconda3\envs\ai-env\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 18s 38ms/step - accuracy: 0.6276 - loss: 0.9276 - val_accuracy: 0.6299 - val_loss: 0.9203
Epoch 2/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 14s 41ms/step - accuracy: 0.6296 - loss: 0.9180 - val_accuracy: 0.6299 - val_loss: 0.9183
Epoch 3/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.6296 - loss: 0.9179 - val_accuracy: 0.6299 - val_loss: 0.9159
Epoch 4/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - accuracy: 0.6296 - loss: 0.9160 - val_accuracy: 0.6299 - val_loss: 0.9152
Epoch 5/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - accuracy: 0.6296 - loss: 0.9151 - val_accuracy: 0.6299 - val_loss: 0.9148


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Better model
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Stop early if model stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

C:\Users\HP\anaconda3\envs\ai-env\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 45s 146ms/step - accuracy: 0.6828 - loss: 0.7519 - val_accuracy: 0.7325 - val_loss: 0.6538
Epoch 2/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 22s 135ms/step - accuracy: 0.8125 - loss: 0.4976 - val_accuracy: 0.7658 - val_loss: 0.6079
Epoch 3/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 23s 138ms/step - accuracy: 0.8714 - loss: 0.3601 - val_accuracy: 0.7658 - val_loss: 0.6670
Epoch 4/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 22s 132ms/step - accuracy: 0.9062 - loss: 0.2788 - val_accuracy: 0.7598 - val_loss: 0.7126


In [8]:
from sklearn.metrics import classification_report
import numpy as np

# Predict
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Convert numbers back to labels
label_names = ['negative', 'neutral', 'positive']

print("=== Bi-LSTM Model ===")
print(classification_report(y_test, y_pred, target_names=label_names))

92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step
=== Bi-LSTM Model ===
              precision    recall  f1-score   support

    negative       0.81      0.92      0.86      1811
     neutral       0.67      0.46      0.55       643
    positive       0.73      0.66      0.69       469

    accuracy                           0.78      2923
   macro avg       0.74      0.68      0.70      2923
weighted avg       0.77      0.78      0.76      2923



In [9]:
import os
print(os.getcwd())

C:\Users\HP\Desktop\sentiment_project
